In [3]:
#Importar librerias y ademas encontrar el pdf por el directorio

from pathlib import Path 
import re 
from pypdf import PdfReader
import pandas as pd 

ruta_pdf =Path("../data/raw/proteccion_usuario_financiero/decalogo_Derechos_Y_Obligaciones.pdf")

print("Ruta completa: ", ruta_pdf.resolve())
print("existe: ",ruta_pdf.exists())

lector = PdfReader(ruta_pdf)
print("Paginas: ",len(lector.pages))




Ruta completa:  C:\Users\gaand\Proyecto-BD2\Proyectos-BD2\data\raw\proteccion_usuario_financiero\decalogo_Derechos_y_Obligaciones.pdf
existe:  True
Paginas:  2


In [4]:
paginas = []
for numero, pagina in enumerate(lector.pages,start = 1):
    texto = pagina.extract_text() or ""
    paginas.append((numero,texto))

for numero,texto in paginas:
    print(f"===== PAGINA {numero} ({len(texto)}) caracteres) =====")
    print(texto)
    print()

===== PAGINA 1 (2717) caracteres) =====
DECÁLOGO PARA LA PARTICIPACIÓN ACTIVA DE LOS 
USUARIOS FINANCIEROS EN EL EJERCICIO DE SUS DERECHOS 
Y CUMPLIMIENTO DE DEBERES 
Constituyen buenas prácticas de los usuarios financieros, en el ejercicio de sus 
derechos y cumplimiento de sus deberes, las siguientes: 
1. Informarse sobre las instituciones que se encuentran debidamente 
autorizadas y supervisadas por la CNBS. 
2. Elegir con plena libertad los productos y servicios financieros que mejor 
respondan a sus necesidades particulares. 
3. Informarse sobre las condiciones generales de los productos y servicios que 
ofrecen las instituciones supervisadas. 
4. Tomar en cuenta las instrucciones y recomendaciones que imparta la 
institución supervisada o la CNBS, sobre los productos o servicios financieros a 
través de los diferentes canales físicos o digitales. 
5. Conocer las condiciones en las que se contrata el servicio o producto 
financiero, requerir y conservar las copias del contrato y t

In [5]:
texto_completo = "\n".join(texto for numero, texto in paginas)
print("caracteres totales:", len(texto_completo))

caracteres totales: 4133


In [6]:
marca = "DECÁLOGO DE OBLIGACIONES"
parte_usuario, parte_instituciones = texto_completo.split(marca)

print("usuario: ", len(parte_usuario), "caracteres")
print("instituciones: ", len(parte_instituciones),"caracteres")
print()
print("--- Final de la parte Usuario ---")
print(parte_usuario[-150:])
print()
print("--- Inicio de la parte instituciones ---")
print(parte_instituciones[:200])

usuario:  1938 caracteres
instituciones:  2171 caracteres

--- Final de la parte Usuario ---
cieros no implica la pérdida o desconocimiento de los derechos que le son 
propios ante las entidades supervisadas y las autoridades competentes. 
  


--- Inicio de la parte instituciones ---
 Y COMPROMISOS QUE ASUMEN 
LAS INSTITUCIONES SUPERVISADAS POR LA COMISIÓN 
NACIONAL DE BANCOS Y SEGUROS (CNBS) 
Las instituciones supervisadas tendrán las siguientes obligaciones, 
comprometiéndose a 


In [7]:
# Cortar cada sección en trozos, justo antes de cada número de punto (1., 2., ... 10.)
puntos_usuario = re.split(r"\n(?=\d{1,2}\.\s)", parte_usuario)
puntos_instituciones = re.split(r"\n(?=\d{1,2}\.\s)", parte_instituciones)

print("trozos usuario:", len(puntos_usuario))
print("trozos instituciones:", len(puntos_instituciones))
print()

# Ver cómo empieza cada trozo de la sección usuario
for i, trozo in enumerate(puntos_usuario):
    print(i, "->", trozo[:60].replace("\n", " "))

trozos usuario: 11
trozos instituciones: 11

0 -> DECÁLOGO PARA LA PARTICIPACIÓN ACTIVA DE LOS  USUARIOS FINAN
1 -> 1. Informarse sobre las instituciones que se encuentran debi
2 -> 2. Elegir con plena libertad los productos y servicios finan
3 -> 3. Informarse sobre las condiciones generales de los product
4 -> 4. Tomar en cuenta las instrucciones y recomendaciones que i
5 -> 5. Conocer las condiciones en las que se contrata el servici
6 -> 6. Informarse sobre las instancias y procedimientos para la 
7 -> 7. Requerir que los productos o servicios, así como los recl
8 -> 8. Evitar sobre endeudarse con las instituciones supervisada
9 -> 9. Suministrar información verdadera, suficiente y oportuna 
10 -> 10. El no ejercicio de las prácticas de protección por parte


In [8]:
# Las dos secciones, sin el trozo 0 (que es solo título e introducción)
secciones = {
    "deberes_usuario": puntos_usuario[1:],
    "obligaciones_instituciones": puntos_instituciones[1:],
}

# Frase que se pone al inicio de cada chunk para dar contexto
titulos = {
    "deberes_usuario": "Deberes y buenas prácticas del usuario financiero",
    "obligaciones_instituciones": "Obligaciones de las instituciones supervisadas por la CNBS",
}

registros = []

for seccion, lista in secciones.items():
    for punto in lista:
        # Separar el número del texto: "3. Permitir al usuario..." -> 3 y "Permitir al usuario..."
        coincidencia = re.match(r"(\d{1,2})\.\s+(.*)", punto, flags=re.DOTALL)
        numero_item = int(coincidencia.group(1))
        texto = coincidencia.group(2)

        # Unir las líneas cortadas: todo salto de línea o espacio repetido pasa a un solo espacio
        texto = re.sub(r"\s+", " ", texto).strip()

        registros.append({
            "documento": ruta_pdf.stem,
            "categoria": ruta_pdf.parent.name,
            "seccion": seccion,
            "numero_item": numero_item,
            "contenido": f"{titulos[seccion]}: {texto}",
        })

print("total de registros:", len(registros))
print()
print(registros[0]["contenido"])
print()
print(registros[12]["contenido"])

total de registros: 20

Deberes y buenas prácticas del usuario financiero: Informarse sobre las instituciones que se encuentran debidamente autorizadas y supervisadas por la CNBS.

Obligaciones de las instituciones supervisadas por la CNBS: Permitir al usuario la consulta gratuita y por los canales que la institución supervisada establezca, en relación a: Estado de sus cuentas, créditos, servicios y productos, incluyendo la información que la institución supervisada reporta a la Central de Información Crediticia (CIC).


In [9]:
# Metadatos que faltan
for i, r in enumerate(registros):
    r["fuente"] = "CNBS - Decálogo de derechos y obligaciones del usuario financiero"
    r["chunk_index"] = i

df = pd.DataFrame(registros)

# Orden de columnas
df = df[["documento", "categoria", "fuente", "seccion", "numero_item", "chunk_index", "contenido"]]

# Guardar
salida = Path("../data/processed/chunks_decalogo.csv")
df.to_csv(salida, index=False, encoding="utf-8")

print("guardado en:", salida.resolve())
print("filas:", len(df))
df.head()

guardado en: C:\Users\gaand\Proyecto-BD2\Proyectos-BD2\data\processed\chunks_decalogo.csv
filas: 20


,documento,categoria,fuente,seccion,numero_item,chunk_index,contenido
0,decalogo_Derechos_Y_Obligaciones,proteccion_usuario_financiero,CNBS - Decálogo de derechos y obligaciones del...,deberes_usuario,1,0,Deberes y buenas prácticas del usuario financi...
1,decalogo_Derechos_Y_Obligaciones,proteccion_usuario_financiero,CNBS - Decálogo de derechos y obligaciones del...,deberes_usuario,2,1,Deberes y buenas prácticas del usuario financi...
2,decalogo_Derechos_Y_Obligaciones,proteccion_usuario_financiero,CNBS - Decálogo de derechos y obligaciones del...,deberes_usuario,3,2,Deberes y buenas prácticas del usuario financi...
3,decalogo_Derechos_Y_Obligaciones,proteccion_usuario_financiero,CNBS - Decálogo de derechos y obligaciones del...,deberes_usuario,4,3,Deberes y buenas prácticas del usuario financi...
4,decalogo_Derechos_Y_Obligaciones,proteccion_usuario_financiero,CNBS - Decálogo de derechos y obligaciones del...,deberes_usuario,5,4,Deberes y buenas prácticas del usuario financi...
